# Homework 2

**Total: 50 points.**

| Part | Content | Points |
|---|---|---:|
| 1 | MHA (5), GQA (5), TP Communication (5), and Paged KV Cache (10) | 25 |
| 2 | Continuous Batching | 20 |
| 3 | MoE Short-Answer Questions | 5 |

**Environment:** Python 3.10+ and PyTorch 2.2+, on CPU. Run the cells from top to bottom. The first setup run downloads the public test data (about 6 MB) from the course GitHub repository.

Replace each `TODO(...)` with code; one task may take several statements. Answer the written questions, run the public checks, and submit this notebook with its outputs.

AI Usage: Using AI is allowed, but make sure you understand the problem.

Please email banruol2@illinois.edu for any questions.

### Setup

Run these cells once. They load the runtime, test data, and public checks. DO NOT EDIT.

In [ ]:
import hashlib
import sys
import types
import urllib.request
from pathlib import Path
import torch

torch.set_num_threads(1)

def _install_module(name, source, **values):
    module = types.ModuleType(name)
    module.__dict__.update(values)
    sys.modules[name] = module
    exec(compile(source, name, "exec"), module.__dict__)


In [ ]:
_FIXTURE_URL = ('https://raw.githubusercontent.com/fanlai0990/CS498/'
                '21ec59c7a206eb6ad464ef47a92dd509ccc49da4/Homework/hw2/attention_cases.pt')
_FIXTURE_SHA256 = '96f147b240ff89adcdfd6de2bb1ca5efa5da0426e981e94e6fe8e7784cc4229b'
_fixture_path = Path('attention_cases.pt')


def _fixture_ok():
    return _fixture_path.exists() and hashlib.sha256(_fixture_path.read_bytes()).hexdigest() == _FIXTURE_SHA256


if not _fixture_ok():
    urllib.request.urlretrieve(_FIXTURE_URL, _fixture_path)
assert _fixture_ok(), f'Downloaded test data does not match; delete {_fixture_path} and rerun this cell.'
_attention_fixtures = torch.load(_fixture_path, weights_only=True)


In [ ]:
_install_module('hw2_runtime', '"""Provided data types, notebook loading, and execution driver for Homework 2."""\nfrom collections import deque\nfrom dataclasses import dataclass, field\nimport json\nfrom pathlib import Path\nimport torch\n\n\n@dataclass\nclass Request:\n    seq_id: str\n    prompt: list[int]\n    max_new_tokens: int\n    generated: list[int] = field(default_factory=list)\n    finished: bool = False\n\n    @property\n    def token_ids(self):\n        return self.prompt + self.generated\n\n\ndef load_definitions(path, namespace):\n    """Load definition cells without executing a notebook\'s examples or checks."""\n    notebook = json.loads(Path(path).read_text())\n    for cell in notebook[\'cells\']:\n        if \'definition\' in cell.get(\'metadata\', {}).get(\'tags\', []):\n            exec(compile(\'\'.join(cell[\'source\']), str(path), \'exec\'), namespace)\n\n\ndef _arrive(scheduler, pending, step):\n    while pending and pending[0][0] <= step:\n        _, request = pending.popleft()\n        scheduler.add(request)\n\n\ndef run(scheduler, execute, requests, arrivals=None):\n    """Drive fresh requests; return one trace row per executed model batch."""\n    times = [0] * len(requests) if arrivals is None else arrivals\n    pending = deque(sorted(zip(times, requests), key=lambda item: item[0]))\n    trace = []\n    step = 0\n    while pending or scheduler.waiting or scheduler.running:\n        _arrive(scheduler, pending, step)\n        if not scheduler.waiting and not scheduler.running:\n            step = pending[0][0]\n            continue\n        selected, is_prefill = scheduler.schedule()\n        tokens = execute(selected, is_prefill)\n        scheduler.postprocess(selected, tokens)\n        trace.append({\'step\': step, \'phase\': \'prefill\' if is_prefill else \'decode\',\n                      \'ids\': tuple(r.seq_id for r in selected), \'tokens\': tuple(tokens)})\n        step += 1\n    return trace\n\n\ndef replay_executor(cache, outputs):\n    """Fixed-token fixture for the public scheduler example; no model computation."""\n    def execute(selected, is_prefill):\n        tokens = []\n        for request in selected:\n            ids = request.token_ids if is_prefill else request.token_ids[-1:]\n            k = torch.tensor(ids, dtype=cache.k_cache.dtype).reshape(-1, 1, 1)\n            k = k.expand(-1, cache.num_kv_heads, cache.head_dim).contiguous()\n            cache.append(request.seq_id, k, -k)\n            tokens.append(outputs[request.seq_id][len(request.generated)])\n        return tokens\n    return execute')


In [ ]:
_install_module('hw2_checks', '"""Public correctness checks. Keep this module and attention_cases.pt beside the notebooks."""\nfrom functools import partial\nfrom pathlib import Path\nimport torch\nfrom hw2_runtime import Request, replay_executor, run\n\n\ndef _suite(title, cases):\n    failures = []\n    for label, check in cases:\n        try:\n            check()\n        except Exception as exc:\n            failures.append(label)\n            print(f\'FAIL {label}: {type(exc).__name__}: {exc}\')\n        else:\n            print(f\'PASS {label}\')\n    if failures:\n        raise AssertionError(f\'{title}: {len(failures)}/{len(cases)} failed: {", ".join(failures)}\')\n    print(f\'{title}: {len(cases)}/{len(cases)} passed.\')\n\n\ndef _close(actual, expected):\n    tolerance = dict(rtol=2e-4, atol=2e-5) if expected.dtype == torch.float32 else dict(rtol=1e-8, atol=1e-10)\n    torch.testing.assert_close(actual, expected, **tolerance)\n\n\ndef _numerical_case(function, row):\n    actual = function(*[x.clone() for x in row[\'args\']])\n    _close(actual, row[\'expected\'])\n\n\ndef _attention_cases(section, function):\n    fixtures = _attention_fixtures\n    return [(f\'{section}: {row["name"]}\', partial(_numerical_case, function, row)) for row in fixtures[section]]\n\n\ndef _causality(function):\n    row = _attention_fixtures[\'mha\'][-1]\n    x, *weights = row[\'args\']\n    changed = x.clone()\n    changed[:, 64:] += 10\n    _close(function(changed, *weights)[:, :64], function(x, *weights)[:, :64])\n\n\ndef check_mha(mha_attention):\n    _suite(\'MHA\', _attention_cases(\'mha\', mha_attention) + [(\'MHA: future-token isolation\', partial(_causality, mha_attention))])\n\n\ndef check_gqa(gqa_attention, grouped_attention):\n    _suite(\'GQA\', _attention_cases(\'gqa\', gqa_attention) + _attention_cases(\'grouped\', grouped_attention))\n\n\ndef _reserve(Cache):\n    c = Cache(3, 2, 2, 3, dtype=torch.float64)\n    c.reserve(\'a\', 2); c.reserve(\'a\', 2); c.reserve(\'b\', 1); c.reserve(\'a\', 3)\n    assert c.block_tables == {\'a\': [0, 2], \'b\': [1]}, c.block_tables\n    assert c.length(\'a\') == c.length(\'b\') == 0\n    before = ({s: list(t) for s, t in c.block_tables.items()}, dict(c.lengths), list(c.free_block_ids))\n    for seq_id, size in [(\'c\', 1), (\'a\', 5)]:\n        try:\n            c.reserve(seq_id, size)\n        except MemoryError:\n            pass\n        else:\n            raise AssertionError(\'Exhausted capacity must raise MemoryError.\')\n        assert before == (c.block_tables, c.lengths, list(c.free_block_ids)), \'Failed reserve changed cache metadata.\'\n    c.reserve(\'empty\', 0)\n    assert c.read(\'empty\')[0].shape == (0, 2, 3)\n\n\ndef _fragmentation(Cache):\n    c = Cache(4, 2, 2, 3, dtype=torch.float64)\n    a = torch.arange(18, dtype=torch.float64).reshape(3, 2, 3)\n    b = a[:2] + 100\n    fresh = a + 200\n    c.append(\'a\', a, -a); c.append(\'b\', b, -b); c.release(\'a\'); c.append(\'c\', fresh, -fresh)\n    assert c.block_tables[\'c\'] == [3, 0], c.block_tables\n    assert torch.equal(c.slot_mapping(\'c\', torch.tensor([3, 0, 2, 1, 1])), torch.tensor([1, 6, 0, 7, 7]))\n    assert c.slot_mapping(\'c\', torch.empty(0, dtype=torch.long)).shape == (0,)\n    _close(c.k_cache[3], fresh[:2]); _close(c.k_cache[0, :1], fresh[2:])\n    _close(c.v_cache[3], -fresh[:2]); _close(c.v_cache[0, :1], -fresh[2:])\n    _close(c.read(\'b\')[0], b); _close(c.read(\'c\')[0], fresh)\n    _close(c.read(\'b\')[1], -b); _close(c.read(\'c\')[1], -fresh)\n\n\ndef _append_oom(Cache):\n    c = Cache(2, 4, 2, 3, dtype=torch.float64)\n    x = torch.arange(24, dtype=torch.float64).reshape(4, 2, 3)\n    c.append(\'a\', x, -x); c.append(\'b\', x + 100, -x - 100)\n    before = (c.k_cache.clone(), c.v_cache.clone())\n    try:\n        c.append(\'a\', x[:1], -x[:1])\n    except MemoryError:\n        pass\n    else:\n        raise AssertionError(\'Append beyond the full pool must raise MemoryError.\')\n    assert c.length(\'a\') == c.length(\'b\') == 4\n    assert c.block_tables == {\'a\': [0], \'b\': [1]}\n    _close(c.k_cache, before[0]); _close(c.v_cache, before[1])\n\n\ndef _append_chunks(c, data, block_size):\n    start = 0\n    for count in [1, block_size - 1, 2 * block_size + 1, block_size]:\n        for seq_id, values in data.items():\n            chunk = values[start:start + count].clone()\n            c.append(seq_id, chunk, -chunk)\n            chunk.fill_(999)\n            assert c.length(seq_id) == start + count\n            k, v = c.read(seq_id)\n            _close(k, values[:start + count]); _close(v, -values[:start + count])\n        start += count\n\n\ndef _reuse_half(c, data):\n    for seq_id in list(data)[::2]:\n        c.release(seq_id); c.release(seq_id)\n        assert seq_id not in c.block_tables and seq_id not in c.lengths\n    for i in range(12):\n        seq_id = f\'reused-{i}\'\n        values = data[str(2 * i)] + 50\n        c.append(seq_id, values, -values)\n        _close(c.read(seq_id)[0], values); _close(c.read(seq_id)[1], -values)\n    for seq_id in list(data)[1::2]:\n        _close(c.read(seq_id)[0], data[seq_id]); _close(c.read(seq_id)[1], -data[seq_id])\n\n\ndef _cache_stress(Cache, block_size, heads, dim, dtype):\n    c = Cache(256, block_size, heads, dim, dtype=dtype)\n    pointers = c.k_cache.data_ptr(), c.v_cache.data_ptr()\n    gen = torch.Generator().manual_seed(601)\n    for _ in range(3):\n        data = {str(i): torch.randn(4 * block_size + 1, heads, dim, generator=gen, dtype=dtype) for i in range(24)}\n        _append_chunks(c, data, block_size)\n        _reuse_half(c, data)\n        owned = [page for table in c.block_tables.values() for page in table]\n        assert len(owned) == len(set(owned)), \'Requests share a physical page.\'\n        assert set(owned).isdisjoint(c.free_block_ids)\n        assert len(owned) + len(c.free_block_ids) == 256\n        for seq_id in list(c.block_tables):\n            c.release(seq_id)\n        assert sorted(c.free_block_ids) == list(range(256)) and not c.lengths\n        assert pointers == (c.k_cache.data_ptr(), c.v_cache.data_ptr())\n\n\ndef check_paged_cache(Cache):\n    cases = [(\'KV: reserve, idempotence, empty sequence, atomic OOM\', partial(_reserve, Cache)),\n             (\'KV: noncontiguous pages, repeated/reordered positions, physical writes\', partial(_fragmentation, Cache)),\n             (\'KV: append OOM preserves both requests\', partial(_append_oom, Cache))]\n    for p, h, d in [(1, 1, 3), (3, 2, 7), (8, 4, 16), (16, 2, 8)]:\n        for dtype in [torch.float32, torch.float64]:\n            cases.append((f\'KV: 24 requests, 3 reuse cycles, P={p}, Hkv={h}, D={d}, {dtype}\', partial(_cache_stress, Cache, p, h, d, dtype)))\n    _suite(\'Paged KV cache\', cases)\n\n\ndef _trace_case(Scheduler, Cache, spec):\n    c = Cache(spec.get(\'blocks\', 256), spec.get(\'page\', 2), 2, 3, dtype=torch.float64)\n    batch, budget, eos = spec.get(\'batch\', 2), spec.get(\'budget\', 64), spec.get(\'eos\', 0)\n    scheduler = Scheduler(c, batch, budget, eos_token_id=eos)\n    requests = [Request(str(i), list(range(1, p + 1)), n) for i, (p, n) in enumerate(spec[\'requests\'])]\n    outputs = spec.get(\'outputs\', {r.seq_id: list(range(100, 100 + r.max_new_tokens)) for r in requests})\n    execute_tokens = replay_executor(c, outputs)\n    calls = 0\n    def execute(selected, prefill):\n        nonlocal calls\n        calls += 1\n        assert calls <= 2 * sum(r.max_new_tokens for r in requests), \'Scheduling did not finish within the call limit.\'\n        assert 0 < len(selected) <= batch, \'Empty batch or exceeded the per-batch request limit.\'\n        assert len({r.seq_id for r in selected}) == len(selected), \'Duplicate request in a batch.\'\n        if prefill:\n            assert sum(len(r.token_ids) for r in selected) <= budget\n        for r in selected:\n            assert c.length(r.seq_id) == (0 if prefill else len(r.token_ids) - 1), \'Wrong phase or incomplete replay.\'\n        tokens = execute_tokens(selected, prefill)\n        for r in selected:\n            k, v = c.read(r.seq_id)\n            expected = torch.tensor(r.token_ids, dtype=k.dtype)[:, None, None].expand_as(k)\n            _close(k, expected); _close(v, -expected)\n        return tokens\n    trace = run(scheduler, execute, requests, arrivals=spec.get(\'arrivals\'))\n    actual = [(r[\'step\'], r[\'phase\'][0], r[\'ids\']) for r in trace]\n    if \'trace\' in spec:\n        assert actual == spec[\'trace\'], f\'Expected trace {spec["trace"]}; got {actual}\'\n    if \'calls\' in spec:\n        assert len(trace) == spec[\'calls\'], f\'Expected {spec["calls"]} calls; got {len(trace)}\'\n    for r in requests:\n        want = outputs[r.seq_id][:r.max_new_tokens]\n        if eos in want:\n            want = want[:want.index(eos) + 1]\n        assert r.generated == want and r.finished, f\'{r.seq_id}: expected {want}, got {r.generated}\'\n    assert not scheduler.waiting and not scheduler.running\n    assert not c.block_tables and not c.lengths\n    assert sorted(c.free_block_ids) == list(range(spec.get(\'blocks\', 256))), \'Lost or duplicated free pages.\'\n\n\ndef _self_preemption_case(Scheduler, Cache, count):\n    blocks = 2 * count - 1\n    c = Cache(blocks, 2, 2, 3)\n    scheduler = Scheduler(c, count, 64, eos_token_id=0)\n    requests = [Request(str(i), [1, 2], 3) for i in range(count)]\n    outputs = {r.seq_id: [10 + 3*i, 11 + 3*i, 12 + 3*i] for i, r in enumerate(requests)}\n    execute = replay_executor(c, outputs)\n    for r in requests:\n        scheduler.add(r)\n    selected, phase = scheduler.schedule()\n    assert phase and selected == requests\n    assert [len(c.block_tables[r.seq_id]) for r in requests] == [1] * count, \'Reserve only the current prefix.\'\n    scheduler.postprocess(selected, execute(selected, phase))\n\n    selected, phase = scheduler.schedule()\n    victim = requests[-1]\n    assert not phase and selected == requests[:-1], \'Keep the successfully reserved decode requests.\'\n    assert list(scheduler.running) == requests[:-1], \'Restore selected requests at the front in order.\'\n    assert list(scheduler.waiting) == [victim], \'Requeue the current candidate when no other victim remains.\'\n    assert victim.generated == outputs[victim.seq_id][:1] and not victim.finished\n    assert c.length(victim.seq_id) == 0 and victim.seq_id not in c.block_tables\n    assert len(c.free_block_ids) == 1\n    scheduler.postprocess(selected, execute(selected, phase))\n    calls = 0\n    def guarded(selected, prefill):\n        nonlocal calls\n        calls += 1\n        assert calls <= 2 * sum(r.max_new_tokens for r in requests), \'Scheduling did not finish within the call limit.\'\n        assert 0 < len(selected) <= count, \'Empty batch or exceeded the per-batch request limit.\'\n        return execute(selected, prefill)\n    trace = run(scheduler, guarded, [])\n    assert [(row[\'phase\'], row[\'ids\']) for row in trace] == [\n        (\'decode\', tuple(r.seq_id for r in requests[:-1])),\n        (\'prefill\', (victim.seq_id,)), (\'decode\', (victim.seq_id,))]\n    assert {r.seq_id: r.generated for r in requests} == outputs\n    assert all(r.finished for r in requests)\n    assert not scheduler.running and not scheduler.waiting\n    assert not c.block_tables and not c.lengths and sorted(c.free_block_ids) == list(range(blocks))\n\n\ndef _decode_self_preemption(Scheduler, Cache):\n    for count in [2, 3]:\n        _self_preemption_case(Scheduler, Cache, count)\n\n\ndef check_continuous_batching(Scheduler, Cache):\n    specs = [\n        (\'resident requests exceed batch limit\', dict(requests=[(1, 2)] * 3,\n            trace=[(0,\'p\',(\'0\',\'1\')), (1,\'p\',(\'2\',)), (2,\'d\',(\'0\',\'1\')), (3,\'d\',(\'2\',))])),\n        (\'full-prefill token budget and FCFS\', dict(batch=4, budget=4, requests=[(3,2),(3,2),(1,2)],\n            trace=[(0,\'p\',(\'0\',)),(1,\'p\',(\'1\',\'2\')),(2,\'d\',(\'0\',\'1\',\'2\'))])),\n        (\'blocked prefill falls back to decode\', dict(blocks=4, requests=[(4,3),(6,1)],\n            trace=[(0,\'p\',(\'0\',)),(1,\'d\',(\'0\',)),(2,\'d\',(\'0\',)),(3,\'p\',(\'1\',))])),\n        (\'tail preemption and full-prefix replay\', dict(blocks=3, batch=1, requests=[(2,3),(2,2),(2,2)],\n            trace=[(0,\'p\',(\'0\',)),(1,\'p\',(\'1\',)),(2,\'p\',(\'2\',)),(3,\'d\',(\'0\',)),(4,\'d\',(\'0\',)),(5,\'p\',(\'2\',)),(6,\'d\',(\'1\',))])),\n        (\'EOS in prefill and decode\', dict(requests=[(1,5),(1,5)], outputs={\'0\':[0], \'1\':[11,0]},\n            trace=[(0,\'p\',(\'0\',\'1\')),(1,\'d\',(\'1\',))])),\n        (\'nonzero EOS and independent length limit\', dict(eos=5, requests=[(1,4),(1,2)], outputs={\'0\':[0,5], \'1\':[0,0]},\n            trace=[(0,\'p\',(\'0\',\'1\')),(1,\'d\',(\'0\',\'1\'))])),\n        (\'arrivals during decode\', dict(requests=[(2,4),(2,1)], arrivals=[0,2],\n            trace=[(0,\'p\',(\'0\',)),(1,\'d\',(\'0\',)),(2,\'p\',(\'1\',)),(3,\'d\',(\'0\',)),(4,\'d\',(\'0\',))])),\n        (\'idle gaps and arrival order\', dict(requests=[(1,2),(1,1)], arrivals=[7,3],\n            trace=[(3,\'p\',(\'1\',)),(7,\'p\',(\'0\',)),(8,\'d\',(\'0\',))])),\n        (\'empty workload\', dict(requests=[], trace=[])),\n        (\'32 requests with equal lengths\', dict(batch=4, page=8, requests=[(16,32)] * 32, calls=256)),\n        (\'64 requests with mixed lengths\', dict(batch=8, page=8, blocks=1024, budget=256, requests=[(17,1),(31,64),(8,7),(32,48)] * 16)),\n        (\'32 requests under repeated cache pressure\', dict(batch=1, page=8, blocks=32, requests=[(16,64),(16,16)] * 16, calls=1280)),\n    ]\n    cases = [(f\'CB: {name}\', partial(_trace_case, Scheduler, Cache, spec)) for name, spec in specs]\n    cases.append((\'CB: current-prefix allocation, candidate self-preemption, and replay\', partial(_decode_self_preemption, Scheduler, Cache)))\n    _suite(\'Continuous batching\', cases)\n', _attention_fixtures=_attention_fixtures)


## Part 1: Attention, TP Communication, and Paged KV Cache (25 points)

**Background.** A decoder-only model generates one token at a time. At each step, attention for the new token needs the keys and values (K/V) of every earlier token, so inference engines store them in a *KV cache* instead of recomputing them. KV-cache memory grows with the number of KV heads, the context length, and the number of concurrent requests, and it limits how many requests a server can run at once. The four sections follow this thread:

1. **1.1 MHA** implements causal attention in which every query head has its own K/V head.
2. **1.2 GQA** lets a group of query heads share one K/V head, which shrinks the KV cache by a factor of `R = Hq / Hkv`.
3. **1.3 TP communication** splits the GQA layer across GPUs and counts the communication this requires.
4. **1.4 Paged KV cache** stores the K/V of many requests in fixed-size pages. The scheduler in Part 2 allocates and frees these pages.

Use `torch.einsum` for the projection and attention contractions in Sections 1.1 and 1.2; other correct implementations are also accepted.

| Symbol | Meaning |
|---|---|
| `B` | Batch size in Sections 1.1 and 1.2 |
| `T`, `S` | Number of query positions and available key/value positions |
| `C` | Embedding dimension |
| `Hq`, `Hkv` | Query heads and KV heads |
| `D` | Head dimension; `C = Hq * D` |
| `R` | Query heads per KV head; `Hq // Hkv` |

Projection weights have shape `[C, number_of_heads, D]`. Attention tensors use `[batch, sequence, head, head_dim]`. Section 1.4 stores already-projected K/V for one request at a time as `[tokens, Hkv, D]`.

In [ ]:
import math
import torch


def TODO(task_id):
    raise NotImplementedError(f"Complete {task_id}.")


In [ ]:
import hw2_checks as public_checks


### 1.1 Multi-Head Attention (5 points)

This is the baseline. Multi-head attention comes from the original Transformer ([Vaswani et al., *Attention Is All You Need*, NeurIPS 2017](https://arxiv.org/abs/1706.03762)) and is used in models such as GPT-3 and the original LLaMA. Causal attention lets each position attend only to itself and earlier positions, which matches token-by-token generation. With `H` query heads, the KV cache stores `H` K/V heads per token.

Implement causal MHA with `Hq = Hkv = H` and weights of shape `[C, H, D]`.

| Task | Points | Computation |
|---|---:|---|
| MHA-1 | 1 | Q, K, and V projections: `[B, T, H, D]`. |
| MHA-2 | 1 | Scaled query-key scores: `[B, H, T, T]`. |
| MHA-3 | 1 | Causal visibility mask: `[T, T]`, including the diagonal. |
| MHA-4 | 1 | Mask future scores and apply softmax over key positions. |
| MHA-5 | 1 | Attention-weighted values: `[B, T, H, D]`. |

Use distinct einsum labels for query and key positions. The final head concatenation is provided. Section 1.2 reuses `project_qkv` and `masked_softmax`.

In [ ]:
def project_qkv(x, w_q, w_k, w_v):
    # MHA-1 (1 point): project Q, K, and V into [B,T,heads,D].
    TODO("MHA-1")
    return q, k, v


def causal_mask(length, device):
    # MHA-3 (1 point): causal visibility, including the diagonal.
    return TODO("MHA-3")


def masked_softmax(scores, allowed):
    # MHA-4 (1 point): exclude masked keys and normalize over the last axis.
    return TODO("MHA-4")


def mha_attention(x, w_q, w_k, w_v):
    B, T, C = x.shape
    H, D = w_q.shape[1:]
    q, k, v = project_qkv(x, w_q, w_k, w_v)
    # MHA-2 (1 point): scaled QK products [B,H,T,T].
    scores = TODO("MHA-2")
    att = masked_softmax(scores, causal_mask(T, x.device))
    # MHA-5 (1 point): attention-weighted values [B,T,H,D].
    heads = TODO("MHA-5")
    return heads.reshape(B, T, C)


#### Public checks

11 cases. Each prints `PASS` or a labeled failure with the mismatched values.

- 10 numerical cases: batch sizes up to 4, sequences up to 128, several head sizes, float32 and float64.
- Causality: changing the inputs at positions 64 and later must not change the outputs at positions 0–63.

In [ ]:
public_checks.check_mha(mha_attention)


### 1.2 Grouped-Query Attention (5 points)

In MHA, each token's KV-cache entry holds `2 * Hq * D` elements per layer. Grouped-query attention ([Ainslie et al., *GQA*, EMNLP 2023](https://arxiv.org/abs/2305.13245)) keeps `Hq` query heads but only `Hkv < Hq` K/V heads, so the entry shrinks to `2 * Hkv * D`. It generalizes multi-query attention ([Shazeer, *Fast Transformer Decoding: One Write-Head is All You Need*, 2019](https://arxiv.org/abs/1911.02150)), which uses a single K/V head (`Hkv = 1`). Llama 2 70B, Llama 3, Mistral 7B, and Qwen2 use GQA. Copying K/V back to `Hq` heads inside attention would give the same numbers but bring back the memory traffic GQA removes, so this section computes attention directly on the compact heads.

Reuse `project_qkv`, `causal_mask`, and `masked_softmax` from Section 1.1. Do not expand K/V to `Hq` heads.

Each contiguous group of `R = Hq // Hkv` query heads shares one KV head. Inputs are `q: [B,T,Hq,D]`, `k/v: [B,S,Hkv,D]`, and `allowed: [T,S]`. `grouped_attention` returns `[B, T, Hq * D]`, with query heads concatenated in their original order and no output projection.

| Task | Required operation | Points |
|---|---|---:|
| GQA-1 | Group queries and compute scaled grouped QK scores | 2 |
| GQA-2 | Broadcast the visibility mask over the grouped score tensor | 1 |
| GQA-3 | Aggregate shared values and restore query-head order | 2 |

Support `Hkv = Hq`, intermediate group counts, and `Hkv = 1`.


In [ ]:
def grouped_attention(q, k, v, allowed):
    """q [B,T,Hq,D], k/v [B,S,Hkv,D], allowed [T,S]; returns [B,T,Hq*D]."""
    B, T, Hq, D = q.shape
    S, Hkv = k.shape[1:3]
    R = Hq // Hkv
    # GQA-1 (2 points): group queries and compute scaled scores [B,Hkv,T,R,S].
    qg = TODO("GQA-1: groups")
    scores = TODO("GQA-1: scores")
    # GQA-2 (1 point): broadcast the [T,S] mask over batch, group, and query-head axes.
    broadcast_mask = TODO("GQA-2")
    att = masked_softmax(scores, broadcast_mask)
    # GQA-3 (2 points): aggregate shared values and restore the original head order.
    heads = TODO("GQA-3: values")
    output = TODO("GQA-3: output")
    return output


def gqa_attention(x, w_q, w_k, w_v):
    q, k, v = project_qkv(x, w_q, w_k, w_v)
    return grouped_attention(q, k, v, causal_mask(x.shape[1], x.device))


#### Public checks

34 cases:

- 26 run `gqa_attention` end to end with MHA (`Hkv = Hq`), MQA (`Hkv = 1`), and GQA head ratios, in float32 and float64.
- 8 call `grouped_attention` directly with unequal query/key lengths (`T != S`), arbitrary visibility masks, and large logits.

In [ ]:
public_checks.check_gqa(gqa_attention, grouped_attention)


### 1.3 GQA Tensor-Parallel Communication (5 points)

Models too large for one GPU split each layer across several GPUs with tensor parallelism (TP), following Megatron-LM ([Shoeybi et al., *Megatron-LM*, 2019](https://arxiv.org/abs/1909.08053)). vLLM, SGLang, and TensorRT-LLM use the same scheme to serve one model on several GPUs. In Megatron-style attention, each GPU computes a subset of the heads, multiplies its output by its slice of the output projection, and a **ring all-reduce** sums these partial results across GPUs. This problem counts that traffic for a GQA layer with the attention dimensions of Llama 3 8B and compares it with MHA.

A written problem: show your calculations. No code is required.

**Layer.** One GQA attention layer followed by an output projection `W_O` (Sections 1.1–1.2 stop before this projection).

| Symbol | Value |
|---|---|
| `Hq` | 32 query heads |
| `Hkv` | 8 KV heads, so `R = 4` query heads share each KV head |
| `D` | 128 |
| `C = Hq * D` | 4096 |
| `P` | 4 GPUs with tensor parallelism (TP) |

For `N` input tokens `x: [N, C]`, the layer projects Q/K/V, runs attention, concatenates the heads into `[N, C]`, and multiplies by `W_O: [C, C]`.

**How the layer is split across GPUs.**

- `x` is already replicated on all four GPUs.
- Heads are split by GQA group. Each GPU owns an equal number of complete groups: their query heads and the KV heads they share. It holds the matching head slices of `W_Q`, `W_K`, and `W_V` (each of shape `[C, heads, D]`, as in Section 1.1).
- Each GPU keeps the KV cache for its own KV heads and computes attention for its own query heads over the full context.
- `W_O` is split by rows (its input dimension). Each GPU holds the rows that multiply its own heads' attention output, so its local product is a partial result for the full layer output.
- The final layer output `[N, C]` must be replicated on all GPUs. The partial results are summed with a ring all-reduce ([Patarasuk and Yuan, 2009](https://doi.org/10.1016/j.jpdc.2008.09.002); see also the [NCCL performance notes](https://github.com/NVIDIA/nccl-tests/blob/master/doc/PERFORMANCE.md#allreduce)). The GPUs form a ring, and the all-reduce runs a reduce-scatter followed by an all-gather. Each phase takes `P - 1` steps; in each step, every GPU sends one chunk (`1/P` of the tensor) to its neighbor.

**Counting rules.**

- Count only activation traffic in this layer's forward pass, from the replicated input to the replicated output. Ignore weight distribution and protocol overhead.
- Activations are BF16: 2 bytes per element.
- **Per-GPU volume** is the number of bytes one GPU sends. **Total volume** is the sum over all GPUs of bytes sent; do not count the same bytes again when they are received.
- `1 KiB = 2^10 bytes` and `1 MiB = 2^20 bytes`.

#### TP-1: Head placement (1 point)

- How many query heads and KV heads does each GPU own?
- Do the Q/K/V projections or the attention computation before `W_O` need any communication between GPUs? Explain using the head placement.

**Answer:**



#### TP-2: Prefill (2 points)

One prefill step processes `B = 8` requests with `L = 512` prompt tokens each, in a single forward pass.

- What are the shape and BF16 size of each GPU's partial result?
- How many bytes does one GPU send in the all-reduce, and how many do all four GPUs send in total? Give both in MiB.

**Answer:**



#### TP-3: Decode and the effect of GQA (2 points)

The same 8 requests now have 2048 cached tokens each. One decode step processes one new token per request.

- How many bytes does one GPU send, and how many do all four GPUs send in total? Give both in KiB.
- Does the 2048-token cached context change this communication volume? Explain.
- Suppose the layer used MHA instead (`Hkv = Hq = 32`), with the same `Hq`, `D`, `C`, `P`, and partitioning scheme. Does GQA cut the TP communication you computed by a factor of four compared with MHA? Explain.

**Answer:**



### 1.4 Paged KV Cache (10 points)

A server runs many requests whose lengths differ and grow by one token per decode step. Reserving a contiguous maximum-length buffer for each request wastes the space that is never filled. PagedAttention ([Kwon et al., *Efficient Memory Management for Large Language Model Serving with PagedAttention*, SOSP 2023](https://arxiv.org/abs/2309.06180)), introduced with vLLM, manages the cache like virtual memory: the cache is a pool of fixed-size blocks, and each request has a block table that maps its logical token positions to physical blocks. Blocks are allocated as a request grows and returned when it finishes, so each request wastes at most the unused part of its last block. SGLang and TensorRT-LLM also manage the KV cache in pages. Part 2's scheduler calls `reserve` and `release` on this cache.

Complete the marked operations in `PagedKVCache`, a block pool shared by all requests.

The fixed CPU pools have shape `[num_blocks, block_size, Hkv, D]`. `free_block_ids` is a FIFO queue; `block_tables[seq_id]` stores physical block IDs in logical order; `lengths[seq_id]` counts written tokens. Assume valid shapes, dtypes, lengths, and positions.

| Task | Required operation | Points |
|---|---|---:|
| KV-1 | Reserve blocks; check capacity before changing any state | 3 |
| KV-2 | Translate logical positions through the page table to physical slots | 3 |
| KV-3 | Append and read K/V through the shared pools | 3 |
| KV-4 | Reclaim blocks and request metadata | 1 |

Keep existing blocks when reserving more. Read only written tokens, including those in a partial last block. Support noncontiguous pages and empty requests. Release blocks in block-table order; repeated release has no effect. `read` raises `KeyError` for an unknown request.

In [ ]:
from abc import ABC, abstractmethod
from collections import deque


class KVCache(ABC):
    """One layer's projected K/V, addressed by a request ID."""
    @abstractmethod
    def append(self, seq_id, new_k, new_v):
        """Append tensors [new_tokens, Hkv, D]; retain all previously written tokens."""
        raise NotImplementedError

    @abstractmethod
    def read(self, seq_id):
        """Return K/V [written_tokens, Hkv, D] in logical sequence order."""
        raise NotImplementedError

    @abstractmethod
    def release(self, seq_id):
        """Reclaim a request's storage; repeated release is a no-op."""
        raise NotImplementedError

    @abstractmethod
    def length(self, seq_id):
        """Return the number of written tokens, or zero for an unknown request."""
        raise NotImplementedError




In [ ]:
class PagedKVCache(KVCache):
    def __init__(self, num_blocks, block_size, num_kv_heads, head_dim,
                 dtype=torch.float32):
        self.block_size = block_size
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim
        shape = (num_blocks, block_size, num_kv_heads, head_dim)
        self.k_cache = torch.empty(shape, dtype=dtype)
        self.v_cache = torch.empty_like(self.k_cache)
        self.free_block_ids = deque(range(num_blocks))
        self.block_tables = {}
        self.lengths = {}

    def length(self, seq_id):
        return self.lengths.get(seq_id, 0)

    def reserve(self, seq_id, total_length):
        current = len(self.block_tables.get(seq_id, []))
        # KV-1 (3 points): count missing blocks, check capacity before changing state,
        # then move free blocks into this request's block table.
        required = TODO("KV-1: required blocks")
        additional = TODO("KV-1: shortfall")
        if TODO("KV-1: insufficient"):
            raise MemoryError("Insufficient KV-cache blocks")
        table = self.block_tables.setdefault(seq_id, [])
        self.lengths.setdefault(seq_id, 0)
        for _ in range(additional):
            TODO("KV-1: transfer block")

    def slot_mapping(self, seq_id, positions):
        table = torch.tensor(self.block_tables[seq_id], dtype=torch.long)
        # KV-2 (3 points): logical block/offset -> physical block -> flat slot.
        logical_blocks = TODO("KV-2: logical blocks")
        offsets = TODO("KV-2: offsets")
        physical_blocks = TODO("KV-2: page-table lookup")
        return TODO("KV-2: slots")

    def append(self, seq_id, new_k, new_v):
        start = self.length(seq_id)
        end = start + new_k.shape[0]
        self.reserve(seq_id, end)
        positions = torch.arange(start, end, dtype=torch.long)
        slots = self.slot_mapping(seq_id, positions)
        k_flat = self.k_cache.flatten(0, 1)
        v_flat = self.v_cache.flatten(0, 1)
        # KV-3 (2 of 3 points): copy both K/V into slots, then commit written length.
        TODO("KV-3: write and commit")

    def read(self, seq_id):
        positions = torch.arange(self.lengths[seq_id], dtype=torch.long)
        slots = self.slot_mapping(seq_id, positions)
        k_flat = self.k_cache.flatten(0, 1)
        v_flat = self.v_cache.flatten(0, 1)
        # KV-3 (1 of 3 points): gather both K/V in logical order.
        k = TODO("KV-3: read keys")
        v = TODO("KV-3: read values")
        return k, v

    def release(self, seq_id):
        if seq_id not in self.block_tables:
            return
        blocks = self.block_tables[seq_id]
        # KV-4 (1 point): return owned blocks and remove both metadata entries.
        TODO("KV-4")


#### Public checks

11 cases:

- `reserve`: repeated reservations, an empty request, and a failed reservation that raises `MemoryError` and leaves all metadata unchanged.
- Fragmented pages: a request whose pages are noncontiguous; `slot_mapping` with repeated and reordered positions; K/V written to the right physical pages.
- An `append` that exceeds the pool raises `MemoryError` without changing either existing request.
- 8 stress runs (4 page-size/head/dim settings, float32 and float64): 24 requests append in chunks that cross page boundaries, half are released, and new requests reuse the freed pages, for 3 cycles. The checks also verify that the `k_cache`/`v_cache` tensors are never reallocated.

In [ ]:
public_checks.check_paged_cache(PagedKVCache)


## Part 2: Continuous Batching (20 points)

**Background.** With *static batching*, a server runs a fixed batch until every request in it finishes. Short requests leave idle batch slots, and new requests wait for the whole batch. *Continuous batching*, or iteration-level scheduling ([Yu et al., *Orca*, OSDI 2022](https://www.usenix.org/conference/osdi22/presentation/yu)), chooses the batch again before every model step: finished requests leave immediately, and waiting requests join as soon as they fit. vLLM, SGLang, Hugging Face TGI, and TensorRT-LLM (where it is called in-flight batching) all schedule this way.

Each request has two phases. *Prefill* processes the whole prompt in one step and writes its K/V into the cache. *Decode* then generates one token per step and appends one K/V entry. As in vLLM's original scheduler without chunked prefill, this assignment never mixes the two phases in one batch and runs a prefill batch whenever one fits.

The KV pool from Section 1.4 is the limiting resource. Every decode step adds one token to each request in the batch, so the pool can run out. The scheduler then *preempts* requests: it frees their blocks and later recomputes their K/V with a new prefill over the full sequence (prompt plus generated tokens). This is vLLM's recomputation-based preemption.

Complete the marked operations in `ContinuousBatcher`, which uses your `PagedKVCache` from Part 1. The provided driver handles execution and request arrivals.

| Task | Points |
|---|---:|
| CB-P: prefill candidate, admission constraints, and queue transition | 6 |
| CB-D: decode candidate, capacity handling, and queue transition | 6 |
| CB-R: preemption and replay state | 2 |
| CB-S: batch-phase selection | 2 |
| CB-F: completion condition and resource reclamation | 4 |

The scheduler keeps `waiting` and `running` queues. Each `schedule()` call returns a prefill batch if one fits, otherwise a decode batch. A batch never mixes prefill and decode.

- Prefill processes the entire current sequence and reserves only its current KV requirement.
- Decode reserves space for one more token. If needed, preempt requests from the back of `running`. If no other request is left to preempt, preempt the current candidate, stop selecting, and run the requests already selected.
- Preemption releases KV and puts the request at the front of `waiting`. Keep its generated tokens. The next prefill rebuilds its full current sequence and produces the next token.
- All requests use the scheduler's integer `eos_token_id`. Finish on this token or the output-token budget, then release KV and remove the request from `running`.

`max_num_seqs` limits requests in a batch; `max_num_batched_tokens` limits tokens in a prefill batch. Assume positive limits and that each request's maximum context fits both the KV pool and the prefill token budget.

In [ ]:
from hw2_runtime import Request, run, replay_executor


#### Interface

`Request.token_ids` is the prompt followed by generated tokens. `Request.generated` stores outputs; `Request.finished` records completion.

- `add(request)`: enqueue a request.
- `schedule()`: return `(selected_requests, is_prefill)`.
- `postprocess(selected, token_ids)`: apply one output token per selected request.

`_try_reserve` returns whether the reservation succeeded. `_make_room` preempts requests from the tail of `running` until the reservation succeeds. It returns `True` on success, or `False` if `running` empties first. On `False`, `_schedule_decode` preempts its current candidate and stops, keeping the requests it already selected.

After selecting a decode batch, put the selected requests back at the **front** of `running` in their original relative order, ahead of any unselected requests. `postprocess` then removes requests that finish.


In [ ]:
from collections import deque


class ContinuousBatcher:
    def __init__(self, cache, max_num_seqs, max_num_batched_tokens, eos_token_id):
        self.cache = cache
        self.max_num_seqs = max_num_seqs
        self.max_num_batched_tokens = max_num_batched_tokens
        self.eos_token_id = eos_token_id
        self.waiting = deque()
        self.running = deque()

    def add(self, request):
        self.waiting.append(request)

    def _try_reserve(self, request, token_count):
        try:
            self.cache.reserve(request.seq_id, token_count)
        except MemoryError:
            return False
        return True

    def _schedule_prefill(self):
        selected = []
        used_tokens = 0
        # CB-P2 (3 points): batch limit, token budget, and reservation.
        while TODO("CB-P2: batch limit"):
            # CB-P1 (1 point): identify the candidate and its current KV requirement.
            request = TODO("CB-P1: candidate")
            token_count = TODO("CB-P1: demand")
            if TODO("CB-P2: token budget"):
                break
            if not TODO("CB-P2: reserve"):
                break
            # CB-P3 (2 points, both queue updates): select the admitted head.
            TODO("CB-P3: select")
            used_tokens += token_count
        TODO("CB-P3: running queue")
        return selected

    def _make_room(self, request, token_count):
        # CB-D2 (3 points, shared with the CB-D2 tasks in `_schedule_decode`): retry, preempting from the tail.
        while not TODO("CB-D2: reserve"):
            if not self.running:
                return False
            self.preempt(TODO("CB-D2: victim"))
        return True

    def _schedule_decode(self):
        selected = []
        # CB-D2: limit this decode batch.
        while TODO("CB-D2: batch limit"):
            # CB-D1 (1 point): identify the candidate and its next-token KV requirement.
            request = TODO("CB-D1: candidate")
            token_count = TODO("CB-D1: demand")
            # CB-D2: if _make_room fails, preempt this candidate and stop selection.
            TODO("CB-D2: handle capacity result")
            # CB-D3 (2 points, both queue updates): select the reserved request.
            TODO("CB-D3: select")
        TODO("CB-D3: running queue")
        return selected

    def schedule(self):
        prefill = self._schedule_prefill()
        # CB-S (2 points): choose the batch and phase. Return (requests, is_prefill).
        TODO("CB-S")

    def preempt(self, request):
        # The caller has removed the request from running.
        # CB-R (2 points): release KV and requeue first, preserving generated tokens.
        TODO("CB-R")

    def postprocess(self, selected, token_ids):
        for request, token in zip(selected, token_ids):
            request.generated.append(token)
            # CB-F1 (2 points): finish on the output budget or the configured EOS.
            if TODO("CB-F1"):
                request.finished = True
                # CB-F2 (2 points): reclaim KV and remove the completed request.
                TODO("CB-F2")


#### Example

`replay_executor` returns fixed tokens instead of running a model. The three prompts fill the cache, so A's decode preempts C; C is rebuilt after A finishes.

In [ ]:
cache = PagedKVCache(3, 2, 2, 2, dtype=torch.float64)
scheduler = ContinuousBatcher(cache, max_num_seqs=1, max_num_batched_tokens=16, eos_token_id=0)
requests = [Request('A', [1, 2], 3), Request('B', [6, 7], 2), Request('C', [11, 12], 2)]
execute = replay_executor(cache, {'A': [3, 4, 5], 'B': [8, 9], 'C': [13, 14]})
trace = run(scheduler, execute, requests)

expected = [
    ('prefill', ('A',)),
    ('prefill', ('B',)),
    ('prefill', ('C',)),
    ('decode', ('A',)),
    ('decode', ('A',)),
    ('prefill', ('C',)),
    ('decode', ('B',)),
]
assert [(row['phase'], row['ids']) for row in trace] == expected
assert [r.generated for r in requests] == [[3, 4, 5], [8, 9], [13, 14]]
assert len(cache.free_block_ids) == 3
for row in trace:
    print(row)


#### Public checks

The 13 cases below cover phase selection, complete-prefill budgets, queue order, blocked-prefill fallback, preemption/replay, EOS, length limits, arrivals, reclamation, and candidate self-preemption. Larger cases run 32–64 requests and up to 1,280 model calls. The fixed-token executor writes to and reads from your cache.

In [ ]:
from hw2_checks import check_continuous_batching
check_continuous_batching(ContinuousBatcher, PagedKVCache)


## Part 3: MoE Short-Answer Questions (5 points)

**Background.** In a Mixture-of-Experts (MoE) layer ([Shazeer et al., *Outrageously Large Neural Networks*, ICLR 2017](https://arxiv.org/abs/1701.06538)), a router sends each token to a few experts. Mixtral 8x7B, DeepSeek-V3, and Qwen3's MoE models use this design. With *expert parallelism* ([Lepikhin et al., *GShard*, 2020](https://arxiv.org/abs/2006.16668)), experts live on different GPUs, so a GPU's work depends on how many tokens its experts receive, and some experts receive many more tokens than others. DeepSeek's expert-parallel load balancer ([EPLB](https://github.com/deepseek-ai/EPLB); redundant experts are described in the [DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437)) replicates heavily used experts and places the replicas so that GPU loads are balanced while communication cost stays low. The five questions follow EPLB's steps: what to balance (Q1), packing expert groups onto nodes (Q2), replicating heavy experts (Q3), ordering the hierarchical steps (Q4), and building the mapping tables the balancer outputs (Q5).

Consider one Mixture-of-Experts (MoE) layer. A *logical expert* is a distinct expert in the model; a *physical replica* is a deployed copy of a logical expert. Assume that processing a token has the same computational cost for every expert. Measure expert load by the number of tokens assigned to it, and assume that a logical expert's load can be divided evenly among its physical replicas.

Answer briefly. For numerical questions, show your assignments or calculations. No code is required.

### Question 1: Load balancing (1 point)

Why does placing the same number of experts on each GPU not necessarily balance computational load? Under the assumptions above, what quantity should be used to measure a GPU's load?

**Answer:**



### Question 2: Balanced packing (1 point)

Four expert groups A, B, C, and D have loads `[90, 70, 30, 10]`, respectively. Assign them to nodes 0 and 1, with exactly two groups per node.

Process groups in descending order of load. Assign each group to the node with the smallest current total load among nodes that are not yet full. Break ties by choosing the node with the smaller ID.

Give the final group assignment and total load of each node.

**Answer:**



### Question 3: Expert replication (1 point)

Three logical experts A, B, and C have loads `[120, 90, 30]`, respectively. Initially, each has one physical replica. Two additional replicas are allocated one at a time.

At each step, replicate the expert with the highest current load per replica, then update its replica count before making the next decision.

Which expert receives each of the two additional replicas, in order? What are the final loads per replica for A, B, and C?

**Answer:**



### Question 4: Hierarchical placement (1 point)

A hierarchical expert load balancer performs these three operations:

- **(i)** Place physical replicas on GPUs within each node.
- **(ii)** Assign expert groups to nodes.
- **(iii)** Replicate experts within each node.

Give the correct execution order. Assume that experts in the same group tend to be accessed together and that communication within a node is faster than communication across nodes. What communication advantage is this placement strategy intended to exploit?

**Answer:**



### Question 5: Logical and physical mappings (1 point)

Physical replicas are numbered 0 through 4. Their logical expert IDs are given by:

```text
physical_to_logical_map = [0, 1, 2, 0, 1]
```

Write both of the following:

- `expert_count`: the number of physical replicas for each logical expert, in logical expert ID order.
- `logical_to_physical_map`: one row per logical expert, listing its physical replica IDs in ascending order. Use two entries per row and fill unused entries with `-1`.

**Answer:**

